# BitcoinGraphGuard — Phase 1: Exploratory Data Analysis (Elliptic++)

**Phase:** 1 of 9 · Exploratory data analysis
**Environment:** Google Colab / Kaggle — this notebook is the executable ML work and never runs on the laptop
**Dataset:** Elliptic++ — Bitcoin transactions + wallet/addresses over 49 time steps
**Outputs:** figures, tables and a JSON digest written to `OUT_DIR`, reviewed afterwards by `docs/EDA.md`

This is the single canonical EDA entrypoint for the project. It exists to answer the questions
the later modelling phases actually depend on:

1. how large and how clean is the dataset, and what are the verified entity/edge counts?
2. how imbalanced are the labels, and how does that imbalance move across the 49 time steps?
3. what do the transaction and wallet features look like, and which of them are redundant?
4. what does the labelled graph look like — degrees, hubs, multi-edges, components?
5. is the proposed temporal split (train 1–34, test 35–49) actually workable?

No model is trained here. No split is finalised and no feature set is frozen; the goal is to
produce the evidence those decisions will be based on.

### How to run

1. Make the Elliptic++ CSVs visible: upload them to Google Drive (Colab) or attach the dataset
   to a Kaggle notebook. The nine expected files are listed in §2.
2. `Runtime → Run all`. The two large feature files (≈695 MB and ≈606 MB) are **streamed in
   chunks**, so a standard 12 GB Colab runtime is enough. Expect a few minutes of wall time.
3. Read the tables and figures saved in `OUT_DIR`, then complete §10–§11 with the numbers this
   run actually produced — nothing in this notebook is pre-filled with expected results.

Raw data is opened **read-only** and is never modified.

## 1. Setup

All configuration lives in one place: dataset location, random seed, chunk size, time-step
constants and output directory. No personal paths are hardcoded — Colab, Kaggle and a local
checkout are resolved automatically, and `ELLIPTIC_DATA_DIR` overrides everything.

In [ ]:
import gc
import json
import os
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import connected_components

SEED = 42                # seeds the systematic sampling and any future stochastic step
CHUNK = 50_000           # rows per streamed CSV chunk
SAMPLE_TARGET = 25_000   # rows kept for quantiles, skewness and correlations
N_STEPS = 49             # Elliptic++ covers 49 contiguous time steps
EARLY_MAX = 34           # candidate training period: steps 1-34
LATE_MIN = 35            # candidate future period: steps 35-49
LABEL_NAMES = {1: "illicit", 2: "licit", 3: "unknown"}

np.random.seed(SEED)
sns.set_theme(context="notebook", style="whitegrid")
plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.titlesize"] = 11

print(f"python {sys.version.split()[0]} | numpy {np.__version__} | pandas {pd.__version__}")

### 1.1 Locate the dataset and the output directory

The raw files live in `Og data/` in the repository, in a Drive folder on Colab, or under
`/kaggle/input` on Kaggle. The cell below picks whichever exists.

In [ ]:
FILES = {
    "txs_features": "txs_features.csv",
    "txs_classes": "txs_classes.csv",
    "txs_edges": "txs_edgelist.csv",
    "wallets_features": "wallets_features.csv",
    "wallets_classes": "wallets_classes.csv",
    "addr_tx": "AddrTx_edgelist.csv",
    "tx_addr": "TxAddr_edgelist.csv",
    "addr_addr": "AddrAddr_edgelist.csv",
}


# One Drive folder holds everything for this project: the raw dataset and every notebook's
# outputs. Change this single line (or set BITCOINGUARD_DRIVE_ROOT) if your layout differs.
DRIVE_ROOT = Path(os.environ.get("BITCOINGUARD_DRIVE_ROOT", "/content/drive/MyDrive/Projects/Bitcoin Graph"))


def resolve_data_dir() -> Path:
    """Find the Elliptic++ CSVs in Colab, Kaggle or a local checkout."""
    override = os.environ.get("ELLIPTIC_DATA_DIR")
    candidates = [
        Path(override) if override else None,
        DRIVE_ROOT / "Dataset",                          # Colab + Drive (project layout)
        Path("/content/drive/MyDrive/elliptic-plus-plus"),   # Colab + Drive
        Path("/content/elliptic-plus-plus"),                 # Colab, uploaded directly
        Path("/kaggle/input/elliptic-plus-plus"),            # Kaggle dataset slug
        Path("/kaggle/input/elliptic"),                      # Kaggle alternate slug
        Path.cwd() / "Og data",                              # local checkout (read-only)
    ]
    for candidate in candidates:
        if candidate is not None and (candidate / FILES["txs_features"]).exists():
            return candidate
    raise FileNotFoundError(
        "Elliptic++ not found. Set ELLIPTIC_DATA_DIR, mount Drive, or add the Kaggle "
        "dataset. Expected files: " + ", ".join(FILES.values())
    )


IN_COLAB = "google.colab" in sys.modules
if IN_COLAB and not Path("/content/drive/MyDrive").exists():
    from google.colab import drive

    drive.mount("/content/drive")

DATA_DIR = resolve_data_dir()
if IN_COLAB and Path("/content/drive/MyDrive").exists():
    OUT_DIR = DRIVE_ROOT / "eda"
elif Path("/kaggle/working").exists():
    OUT_DIR = Path("/kaggle/working/eda")
else:
    OUT_DIR = Path.cwd() / "reports" / "eda_notebook"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("dataset :", DATA_DIR)
print("outputs :", OUT_DIR)

### 1.2 Small helpers

`REFERENCE` holds the counts verified in Phase 1 (`docs/DATA_INVENTORY.md`). They are used only
as an assertion baseline — the notebook re-derives every one of them, and anything that does not
match prints a `!!` warning instead of being quietly accepted.

In [ ]:
TX_ID, ADDRESS, STEP = "txId", "address", "Time step"

REFERENCE = {
    "txs": 203_769,
    "wallets": 822_942,
    "txs_illicit": 4_545,
    "txs_licit": 42_019,
    "txs_unknown": 157_205,
    "wallet_illicit": 14_266,
    "wallet_licit": 251_088,
    "wallet_unknown": 557_588,
    "txs_edges": 234_355,
    "addr_tx_edges": 477_117,
    "tx_addr_edges": 837_124,
    "addr_addr_edges": 2_868_964,
    "wallets_raw_rows": 1_268_260,
    "wallets_distinct_snapshots": 920_691,
    "txs_blank_rows": 965,
}

# One entry per heterogeneous edge type: which file, which columns, and which node
# universe the endpoints belong to ("combined" = transactions + wallets in one index).
EDGE_SPECS = {
    "tx->tx": {"key": "txs_edges", "source": "txId1", "target": "txId2", "universe": "txs"},
    "addr->tx": {"key": "addr_tx", "source": "input_address", "target": TX_ID, "universe": "combined"},
    "tx->addr": {"key": "tx_addr", "source": TX_ID, "target": "output_address", "universe": "combined"},
    "addr->addr": {"key": "addr_addr", "source": "input_address", "target": "output_address", "universe": "wallets"},
}

CHECKS: list = []


def check(name: str, value, expected=None, note: str = "") -> None:
    """Record a sanity check so §10 can report exactly what was verified."""
    ok = expected is None or value == expected

    def readable(item):
        return item if isinstance(item, (int, float, bool, str, type(None))) else str(item)

    CHECKS.append({"check": name, "value": readable(value), "expected": readable(expected), "ok": bool(ok)})
    if not ok:
        print(f"  !! {name}: got {value}, expected {expected} {note}".rstrip())


def read_header(path: Path) -> list:
    """Column names of a CSV without reading any data rows."""
    return list(pd.read_csv(path, nrows=0).columns)


def iter_chunks(path: Path, usecols=None, chunksize: int = CHUNK):
    """Stream a CSV in chunks; the whole file is never held in memory at once."""
    yield from pd.read_csv(path, usecols=usecols, chunksize=chunksize, low_memory=True)


def count_rows(path: Path) -> int:
    """Count data rows by streaming a single column."""
    total = 0
    for chunk in pd.read_csv(path, usecols=[0], chunksize=200_000):
        total += len(chunk)
    return total


def unique_ids(path: Path, column: str) -> set:
    """Distinct values of one id column, streamed (used for cross-file integrity)."""
    seen: set = set()
    for chunk in iter_chunks(path, usecols=[column], chunksize=200_000):
        seen.update(chunk[column].astype(str).to_numpy())
    return seen


def jsonable(obj):
    """Recursively convert numpy/pandas scalars so results can be serialised."""
    if isinstance(obj, dict):
        return {str(k): jsonable(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [jsonable(v) for v in obj]
    if isinstance(obj, np.integer):
        return int(obj)
    if isinstance(obj, np.floating):
        value = float(obj)
        return value if np.isfinite(value) else None
    if isinstance(obj, np.ndarray):
        return jsonable(obj.tolist())
    if isinstance(obj, float) and not np.isfinite(obj):
        return None
    return obj


def save_json(name: str, payload) -> None:
    (OUT_DIR / name).write_text(json.dumps(jsonable(payload), indent=2, sort_keys=True), encoding="utf-8")


def save_table(name: str, frame: pd.DataFrame) -> pd.DataFrame:
    frame.to_csv(OUT_DIR / name, index=False)
    return frame


def show(frame: pd.DataFrame, decimals: int = 3) -> None:
    """Display a frame with numeric columns rounded, so mixed-type tables stay readable."""
    rounded = frame.copy()
    numeric = rounded.select_dtypes("number").columns
    if len(numeric):
        rounded[numeric] = rounded[numeric].round(decimals)
    display(rounded)


def save_fig(fig, name: str) -> None:
    fig.tight_layout()
    fig.savefig(OUT_DIR / name, dpi=130, bbox_inches="tight")
    plt.show()


save_json("config.json", {
    "seed": SEED, "chunk_size": CHUNK, "sample_target": SAMPLE_TARGET, "n_steps": N_STEPS,
    "early_max": EARLY_MAX, "late_min": LATE_MIN, "data_dir": str(DATA_DIR), "output_dir": str(OUT_DIR),
    "python": sys.version.split()[0], "numpy": np.__version__, "pandas": pd.__version__,
})

### 1.3 Two label conventions used everywhere below

Both are easy to get wrong and both would silently invalidate a later experiment:

* class `3` (**unknown**) is *not* licit — it is counted separately and never folded into the
  legitimate class;
* **"labeled"** always means class `1` or `2` only.

Everything that varies with time is accumulated as a length-50 counter array (index `0` unused,
indices `1..49` = time steps).

In [ ]:
def label_arrays(steps, labels) -> dict:
    """Per-time-step counts split by class. ``labels`` may contain NaN."""
    steps = np.asarray(steps, dtype=np.int64)
    labels = np.asarray(labels, dtype="float64")
    assert steps.size == 0 or steps.max() <= N_STEPS, "unexpected time step value"

    def counts(mask):
        return np.bincount(steps[mask], minlength=N_STEPS + 1)[: N_STEPS + 1]

    return {
        "total": counts(np.ones(steps.shape, dtype=bool)),
        "illicit": counts(labels == 1),
        "licit": counts(labels == 2),
        "unknown": counts(labels == 3),
        "unlabeled": counts(~np.isin(labels, [1, 2, 3])),
    }


def add_counts(dest: dict, source: dict) -> None:
    """Accumulate per-step counters across chunks."""
    for key, values in source.items():
        dest[key] = dest.get(key, np.zeros(N_STEPS + 1, dtype=np.int64)) + values


def step_table(counts: dict, unit: str) -> pd.DataFrame:
    """Turn the counter arrays into the tidy per-time-step table used throughout."""
    frame = pd.DataFrame({
        "time_step": np.arange(1, N_STEPS + 1),
        "unit": unit,
        "total": counts["total"][1:],
        "illicit": counts["illicit"][1:],
        "licit": counts["licit"][1:],
        "unknown": counts["unknown"][1:],
    })
    frame["labeled"] = frame["illicit"] + frame["licit"]
    frame["illicit_pct_of_labeled"] = 100 * frame["illicit"] / frame["labeled"].replace(0, np.nan)
    frame["labeled_pct_of_total"] = 100 * frame["labeled"] / frame["total"].replace(0, np.nan)
    return frame

### 1.4 Streaming feature statistics

The two feature files are the only large inputs. Each is read **once**, in 50k-row chunks, and
reduced to compact accumulators instead of a DataFrame:

* exact whole-file counters for count / missing / mean / std / min / max / zero / negative / infinite;
* moments conditioned on the label (illicit vs licit), giving a point-biserial correlation with illicit;
* moments conditioned on the period (steps 1–34 vs 35–49) to measure temporal drift;
* a bounded systematic sample (~25k rows) for quantiles, skewness and pairwise correlations,
  which cannot be accumulated exactly in one pass.

Nothing here is fitted or selected — §7 only reports what the data looks like.

In [ ]:
class FeatureStats:
    """Streaming per-column statistics for one node type."""

    def __init__(self, columns, stride: int):
        n = len(columns)
        self.columns = list(columns)
        self.stride = max(1, int(stride))
        self._seen = 0
        self._sample: list = []
        self._sample_labels: list = []
        zeros = np.zeros(n)
        self.count = zeros.copy()
        self.missing = zeros.copy()
        self.total = zeros.copy()
        self.total_sq = zeros.copy()
        self.zeros = zeros.copy()
        self.negatives = zeros.copy()
        self.inf = zeros.copy()
        self.lo = np.full(n, np.inf)
        self.hi = np.full(n, -np.inf)
        self.by_label = {label: {"n": zeros.copy(), "s": zeros.copy(), "ss": zeros.copy()} for label in (1, 2)}
        self.early = {"n": zeros.copy(), "s": zeros.copy()}
        self.late = {"n": zeros.copy(), "s": zeros.copy()}

    def update(self, frame: pd.DataFrame, labels, steps) -> None:
        values = frame[self.columns].to_numpy(dtype="float64", na_value=np.nan)
        missing = np.isnan(values)
        present = ~missing
        filled = np.where(missing, 0.0, values)

        self.missing += missing.sum(axis=0)
        self.count += present.sum(axis=0)
        self.total += filled.sum(axis=0)
        self.total_sq += (filled * filled).sum(axis=0)
        self.zeros += ((filled == 0) & present).sum(axis=0)
        self.negatives += ((filled < 0) & present).sum(axis=0)
        self.inf += np.isinf(filled).sum(axis=0)
        self.lo = np.minimum(self.lo, np.where(present, values, np.inf).min(axis=0))
        self.hi = np.maximum(self.hi, np.where(present, values, -np.inf).max(axis=0))

        for label, block in self.by_label.items():
            mask = labels == label
            if mask.any():
                observed = present[mask]
                block["n"] += observed.sum(axis=0)
                block["s"] += np.where(observed, values[mask], 0.0).sum(axis=0)
                block["ss"] += (np.where(observed, values[mask], 0.0) ** 2).sum(axis=0)

        early = steps <= EARLY_MAX
        late = steps >= LATE_MIN
        if early.any():
            self.early["n"] += present[early].sum(axis=0)
            self.early["s"] += np.where(present[early], values[early], 0.0).sum(axis=0)
        if late.any():
            self.late["n"] += present[late].sum(axis=0)
            self.late["s"] += np.where(present[late], values[late], 0.0).sum(axis=0)

        picks = np.arange(self._seen, self._seen + len(frame))
        self._seen += len(frame)
        selected = np.flatnonzero(picks % self.stride == 0)
        if selected.size:
            self._sample.append(frame.iloc[selected][self.columns].to_numpy(dtype="float32"))
            self._sample_labels.append(np.asarray(labels)[selected])

    def sample(self) -> pd.DataFrame:
        if not self._sample:
            return pd.DataFrame(columns=self.columns)
        return pd.DataFrame(np.vstack(self._sample), columns=self.columns)

    def sample_labels(self) -> np.ndarray:
        return np.concatenate(self._sample_labels) if self._sample_labels else np.array([])

    def mean_by_label(self, label: int) -> np.ndarray:
        block = self.by_label[label]
        with np.errstate(invalid="ignore", divide="ignore"):
            return block["s"] / np.where(block["n"] > 0, block["n"], np.nan)

    def corr_with_illicit(self) -> np.ndarray:
        """Point-biserial correlation of the feature with "is illicit".

        Positive means higher values for illicit nodes. The scale is the SD of all labelled
        nodes, so the value stays in [-1, 1]; dividing by the pooled *within-group* SD
        instead would give Cohen's d and can exceed 1.
        """
        n_illicit, n_licit = self.by_label[1]["n"], self.by_label[2]["n"]
        mean_illicit, mean_licit = self.mean_by_label(1), self.mean_by_label(2)
        n_labeled = n_illicit + n_licit
        with np.errstate(invalid="ignore", divide="ignore"):
            labeled_sum = self.by_label[1]["s"] + self.by_label[2]["s"]
            labeled_sum_sq = self.by_label[1]["ss"] + self.by_label[2]["ss"]
            labeled_mean = labeled_sum / n_labeled
            labeled_var = np.maximum(labeled_sum_sq / n_labeled - labeled_mean**2, 0)
            return (
                (mean_illicit - mean_licit)
                / np.sqrt(labeled_var)
                * np.sqrt(n_illicit * n_licit)
                / n_labeled
            )

    def summary(self) -> pd.DataFrame:
        with np.errstate(invalid="ignore", divide="ignore"):
            mean = self.total / self.count
            std = np.sqrt(np.maximum(self.total_sq / self.count - mean**2, 0))
            frame = pd.DataFrame({
                "column": self.columns,
                "count": self.count.astype(np.int64),
                "missing": self.missing.astype(np.int64),
                "missing_pct": 100 * self.missing / (self.count + self.missing),
                "mean": mean,
                "std": std,
                "min": self.lo,
                "max": self.hi,
                "zeros_pct": 100 * self.zeros / self.count,
                "negatives_pct": 100 * self.negatives / self.count,
                "infinite": self.inf.astype(np.int64),
                "mean_early_1_34": self.early["s"] / self.early["n"],
                "mean_late_35_49": self.late["s"] / self.late["n"],
                "corr_with_illicit": self.corr_with_illicit(),
            })
        frame["drift_std"] = (frame["mean_late_35_49"] - frame["mean_early_1_34"]) / frame["std"].replace(0, np.nan)
        frame["constant"] = frame["min"] == frame["max"]
        frame["near_constant"] = frame["constant"] | (frame["zeros_pct"] >= 99)
        return frame

    def release(self) -> None:
        self._sample.clear()
        self._sample_labels.clear()
        gc.collect()

## 2. Load Data

Three tiers, chosen by size:

| Tier | Files | How |
| :--- | :--- | :--- |
| Tiny (labels) | `txs_classes.csv`, `wallets_classes.csv` | loaded fully — 203k / 823k rows |
| Small (edges) | the four edge lists | streamed; never held as one frame |
| Large (features) | `txs_features.csv` (≈695 MB), `wallets_features.csv` (≈606 MB) | streamed **once each**, reduced to accumulators |

`wallets_features_classes_combined.csv` is deliberately not read: it is `wallets_features` plus
the `class` column, byte-for-byte the same rows (verified in Phase 1), so reading it would double
the wallet cost for no new information.

In [ ]:
inventory = []
for key, filename in FILES.items():
    path = DATA_DIR / filename
    header = read_header(path)
    inventory.append({
        "key": key,
        "file": filename,
        "size_mb": round(path.stat().st_size / 1e6, 2),
        "columns": len(header),
        "id_columns": [c for c in header
                       if c in (TX_ID, ADDRESS, STEP, "txId1", "txId2", "input_address", "output_address")],
    })
inventory = pd.DataFrame(inventory)
save_table("file_inventory.csv", inventory)
print(f"raw data: {inventory['size_mb'].sum():,.0f} MB across {len(inventory)} files")
show(inventory)

In [ ]:
txs_classes = pd.read_csv(DATA_DIR / FILES["txs_classes"], dtype={TX_ID: str})
wallets_classes = pd.read_csv(DATA_DIR / FILES["wallets_classes"], dtype={ADDRESS: str})

for entity, frame, key, expected_rows in (
    ("transactions", txs_classes, TX_ID, REFERENCE["txs"]),
    ("wallets", wallets_classes, ADDRESS, REFERENCE["wallets"]),
):
    check(f"{entity}: rows match Phase 1", len(frame), expected_rows)
    check(f"{entity}: duplicate ids", int(frame[key].duplicated().sum()), 0)
    check(f"{entity}: blank ids", int(frame[key].isna().sum()), 0)
    check(f"{entity}: label values are a subset of 1/2/3", bool(frame["class"].isin([1, 2, 3]).all()), True)

print(txs_classes.head(3).to_string(index=False))
print(f"\ntransactions: {len(txs_classes):,}   wallets: {len(wallets_classes):,}")

### 2.1 Transaction feature pass (streamed, single read)

One pass over `txs_features.csv` produces everything the later sections need from it: the
per-time-step label activity, the feature statistics, the `txId → Time step` map, and the set of
transactions whose 17 domain columns are blank (the address-less transactions Phase 1 found).

In [ ]:
txs_path = DATA_DIR / FILES["txs_features"]
txs_columns = read_header(txs_path)
txs_features_all = [c for c in txs_columns if c not in (TX_ID, STEP)]
txs_domain_columns = [c for c in txs_features_all
                      if not c.startswith(("Local_feature", "Aggregate_feature"))]
print(f"{len(txs_features_all)} feature columns = {len(txs_domain_columns)} domain "
      f"+ {len(txs_features_all) - len(txs_domain_columns)} anonymous (Local/Aggregate)")

txs_probe = next(iter_chunks(txs_path, chunksize=5_000))
txs_non_numeric = [c for c in txs_features_all if not pd.api.types.is_numeric_dtype(txs_probe[c])]
check("transactions: non-numeric feature columns", txs_non_numeric, [])
del txs_probe
gc.collect()

txs_stride = max(1, REFERENCE["txs"] // SAMPLE_TARGET)
txs_class_map = dict(zip(txs_classes[TX_ID], txs_classes["class"]))
txs_counts: dict = {}
txs_stats = FeatureStats(txs_features_all, txs_stride)
txs_meta_parts: list = []
blank_tx_ids: set = set()
txs_rows = 0
started = time.time()
for chunk in iter_chunks(txs_path):
    ids = chunk[TX_ID].astype(str)
    labels = ids.map(txs_class_map).to_numpy(dtype="float64", na_value=np.nan)
    steps = chunk[STEP].to_numpy(dtype=np.int64)
    add_counts(txs_counts, label_arrays(steps, labels))
    txs_stats.update(chunk, labels, steps)
    txs_meta_parts.append(pd.DataFrame({TX_ID: ids.to_numpy(), STEP: steps}))
    blank_rows = chunk[txs_domain_columns].isna().any(axis=1)
    blank_tx_ids.update(ids[blank_rows].to_numpy())
    txs_rows += len(chunk)
    del chunk, ids, labels, steps
    gc.collect()

txs_meta = pd.concat(txs_meta_parts, ignore_index=True)
del txs_meta_parts
gc.collect()
print(f"streamed {txs_rows:,} transaction rows in {time.time() - started:.0f}s | "
      f"blank domain columns: {len(blank_tx_ids):,} rows")

check("transactions: feature rows", txs_rows, REFERENCE["txs"])
check("transactions: feature rows with no class label", int(txs_counts["unlabeled"][1:].sum()), 0)
check("transactions: rows with blank domain features", len(blank_tx_ids), REFERENCE["txs_blank_rows"])

### 2.2 Wallet feature pass (streamed, de-duplicated)

The wallet file is the one place the raw data needs correcting before analysis. Phase 1 found
**347,569 exact duplicate rows** — 1,268,260 raw rows collapse to 920,691 distinct
`(address, time step)` snapshots. Duplicates are mostly adjacent, so de-duplication has to happen
*within* a chunk as well as across chunks; the pass below does both and reports the correction
instead of silently absorbing it.

Snapshot keys are kept in a set so a repeated `(address, time step)` in a later chunk is also
caught. That set is the largest object this notebook holds (~920k short strings) and is released
immediately afterwards.

In [ ]:
wallets_path = DATA_DIR / FILES["wallets_features"]
wallets_columns = read_header(wallets_path)
wallets_features_all = [c for c in wallets_columns if c not in (ADDRESS, STEP)]

wallets_probe = next(iter_chunks(wallets_path, chunksize=5_000))
wallets_non_numeric = [c for c in wallets_features_all if not pd.api.types.is_numeric_dtype(wallets_probe[c])]
check("wallets: non-numeric feature columns", wallets_non_numeric, [])
del wallets_probe
gc.collect()

wallet_stride = max(1, REFERENCE["wallets_distinct_snapshots"] // SAMPLE_TARGET)
wallet_class_map = dict(zip(wallets_classes[ADDRESS], wallets_classes["class"]))
wallet_counts: dict = {}
wallet_stats = FeatureStats(wallets_features_all, wallet_stride)
wallet_raw_rows = wallet_distinct_rows = wallet_duplicate_rows = wallet_blank_cells = 0
seen_keys: set = set()
seen_addresses: set = set()
started = time.time()
for chunk in iter_chunks(wallets_path):
    keys = chunk[ADDRESS].astype(str) + "|" + chunk[STEP].astype(str)
    wallet_raw_rows += len(chunk)
    keep = ~keys.isin(seen_keys) & ~keys.duplicated()
    wallet_duplicate_rows += len(chunk) - int(keep.sum())
    if keep.any():
        unique = chunk.loc[keep].copy()
        seen_keys.update(keys[keep].to_numpy())
        addresses = unique[ADDRESS].astype(str)
        seen_addresses.update(addresses.to_numpy())
        unique_steps = unique[STEP].to_numpy(dtype=np.int64)
        unique_labels = addresses.map(wallet_class_map).to_numpy(dtype="float64", na_value=np.nan)
        add_counts(wallet_counts, label_arrays(unique_steps, unique_labels))
        wallet_stats.update(unique, unique_labels, unique_steps)
        wallet_blank_cells += int(unique[wallets_features_all].isna().sum().sum())
        wallet_distinct_rows += len(unique)
        del unique, addresses, unique_steps, unique_labels
    del chunk, keys, keep
    gc.collect()

print(f"streamed {wallet_raw_rows:,} wallet rows in {time.time() - started:.0f}s -> "
      f"{wallet_distinct_rows:,} distinct snapshots ({wallet_duplicate_rows:,} duplicates dropped)")

check("wallets: raw rows", wallet_raw_rows, REFERENCE["wallets_raw_rows"])
check("wallets: distinct (address, time step) snapshots", wallet_distinct_rows, REFERENCE["wallets_distinct_snapshots"])
check("wallets: duplicates dropped", wallet_duplicate_rows,
      REFERENCE["wallets_raw_rows"] - REFERENCE["wallets_distinct_snapshots"])
check("wallets: distinct addresses", len(seen_addresses), REFERENCE["wallets"])
check("wallets: feature rows with no class label", int(wallet_counts["unlabeled"][1:].sum()), 0)
check("wallets: blank feature cells", wallet_blank_cells, 0)

seen_keys.clear()
seen_addresses.clear()
del seen_keys, seen_addresses
gc.collect()

## 3. Dataset Overview

Entity counts, dimensionality, edge inventory and label availability — all re-derived here and
shown next to the Phase 1 reference values. `rows` is the number of raw rows actually parsed for
each entity; for wallets it is the de-duplicated snapshot count, not the raw row count.

In [ ]:
label_counts = {
    "transactions": {int(k): int(v) for k, v in txs_classes["class"].value_counts().items()},
    "wallets": {int(k): int(v) for k, v in wallets_classes["class"].value_counts().items()},
}

overview = pd.DataFrame([
    {"entity": "transactions", "nodes": len(txs_classes), "feature_columns": len(txs_features_all),
     "rows": txs_rows, "time_steps": N_STEPS,
     "illicit": label_counts["transactions"].get(1, 0),
     "licit": label_counts["transactions"].get(2, 0),
     "unknown": label_counts["transactions"].get(3, 0)},
    {"entity": "wallets", "nodes": len(wallets_classes), "feature_columns": len(wallets_features_all),
     "rows": wallet_distinct_rows, "time_steps": N_STEPS,
     "illicit": label_counts["wallets"].get(1, 0),
     "licit": label_counts["wallets"].get(2, 0),
     "unknown": label_counts["wallets"].get(3, 0)},
])
overview["labeled"] = overview["illicit"] + overview["licit"]
overview["labeled_pct"] = 100 * overview["labeled"] / overview["nodes"]
overview["illicit_pct_of_labeled"] = 100 * overview["illicit"] / overview["labeled"]
save_table("overview.csv", overview)

edge_inventory = pd.DataFrame([
    {"edge_type": name, "file": FILES[spec["key"]],
     "source": spec["source"], "target": spec["target"],
     "rows": count_rows(DATA_DIR / FILES[spec["key"]])}
    for name, spec in EDGE_SPECS.items()
])
save_table("edge_inventory.csv", edge_inventory)

check("transactions: illicit count", int(overview.loc[0, "illicit"]), REFERENCE["txs_illicit"])
check("transactions: licit count", int(overview.loc[0, "licit"]), REFERENCE["txs_licit"])
check("transactions: unknown count", int(overview.loc[0, "unknown"]), REFERENCE["txs_unknown"])
check("wallets: illicit count", int(overview.loc[1, "illicit"]), REFERENCE["wallet_illicit"])
check("wallets: licit count", int(overview.loc[1, "licit"]), REFERENCE["wallet_licit"])
check("wallets: unknown count", int(overview.loc[1, "unknown"]), REFERENCE["wallet_unknown"])
check("edges: rows per type", edge_inventory.set_index("edge_type")["rows"].to_dict(),
      {"tx->tx": REFERENCE["txs_edges"], "addr->tx": REFERENCE["addr_tx_edges"],
       "tx->addr": REFERENCE["tx_addr_edges"], "addr->addr": REFERENCE["addr_addr_edges"]})

show(overview)
show(edge_inventory)

## 4. Data Quality

Four things that would silently corrupt a later experiment: missing values, duplicate entities,
identifiers that do not reconcile across files, and columns that carry no information.

In [ ]:
txs_quality = txs_stats.summary()
wallet_quality = wallet_stats.summary()

for entity, summary in (("transactions", txs_quality), ("wallets", wallet_quality)):
    print(f"{entity}: {int((summary['missing'] > 0).sum())} columns with missing values | "
          f"{int(summary['infinite'].sum())} infinite values | "
          f"{int(summary['constant'].sum())} constant | {int(summary['near_constant'].sum())} near-constant")

print("\ntransactions - columns with missing values:")
display(txs_quality.loc[txs_quality["missing"] > 0,
                        ["column", "count", "missing", "missing_pct"]].head(20))
print("wallets - columns with missing values:")
display(wallet_quality.loc[wallet_quality["missing"] > 0,
                           ["column", "count", "missing", "missing_pct"]].head(20))

### 4.1 Do identifiers reconcile across files?

Every transaction and address appearing in an edge list should exist in the corresponding label
file. The 965 transactions with blank domain columns should be exactly the transactions that
appear in no address link — if those two independently-derived sets agree, the missingness is
explained by structure rather than being random corruption.

In [ ]:
txs_ids_all = set(txs_classes[TX_ID])
wallet_ids_all = set(wallets_classes[ADDRESS])

txs_with_links = (unique_ids(DATA_DIR / FILES["addr_tx"], TX_ID)
                  | unique_ids(DATA_DIR / FILES["tx_addr"], TX_ID))
addresses_in_edges = (unique_ids(DATA_DIR / FILES["addr_tx"], "input_address")
                      | unique_ids(DATA_DIR / FILES["tx_addr"], "output_address")
                      | unique_ids(DATA_DIR / FILES["addr_addr"], "input_address")
                      | unique_ids(DATA_DIR / FILES["addr_addr"], "output_address"))

check("transactions: edge endpoints exist in txs_classes", txs_with_links <= txs_ids_all, True)
check("transactions: with at least one address link", len(txs_with_links),
      REFERENCE["txs"] - REFERENCE["txs_blank_rows"])
check("blank-domain rows == address-less transactions", blank_tx_ids == txs_ids_all - txs_with_links, True)
check("wallets: addresses in edge lists == wallets_classes", addresses_in_edges == wallet_ids_all, True)

print(f"transactions with address links : {len(txs_with_links):,} / {len(txs_ids_all):,}")
print(f"transactions without links      : {len(txs_ids_all - txs_with_links):,}")
print(f"addresses covered by edge lists : {len(addresses_in_edges):,} / {len(wallet_ids_all):,}")

del txs_with_links, addresses_in_edges, txs_ids_all, wallet_ids_all
gc.collect()

In [ ]:
# Constant / near-constant columns and impossible values (negative counts, infinities).
for entity, summary in (("transactions", txs_quality), ("wallets", wallet_quality)):
    print(f"--- {entity} ---")
    print("  constant or near-constant :",
          summary.loc[summary["near_constant"], "column"].tolist() or "none")
    print("  infinite values           :",
          summary.loc[summary["infinite"] > 0, "column"].tolist() or "none")
    count_like = summary[summary["column"].str.contains("degree|num_", regex=True)]
    print("  count-like, negative      :",
          count_like.loc[count_like["negatives_pct"] > 0, "column"].tolist() or "none")
    print(f"  columns with any zeros    : {int((summary['zeros_pct'] > 0).sum())}")

## 5. Class Distribution

Transactions and wallets are labelled separately, and the two are not the same population: wallet
labels are per address and reused across every snapshot that address appears in. All three views
below are therefore reported side by side, always keeping `unknown` out of the legitimate class.

In [ ]:
def class_distribution(frame: pd.DataFrame, entity: str) -> dict:
    counts = {int(k): int(v) for k, v in frame["class"].value_counts().items()}
    total = len(frame)
    labeled = counts.get(1, 0) + counts.get(2, 0)
    return {
        "entity": entity,
        "total": total,
        "illicit": counts.get(1, 0),
        "licit": counts.get(2, 0),
        "unknown": counts.get(3, 0),
        "labeled": labeled,
        "labeled_pct": 100 * labeled / total,
        "illicit_pct_of_all": 100 * counts.get(1, 0) / total,
        "illicit_pct_of_labeled": 100 * counts.get(1, 0) / labeled,
        "licit_per_illicit": counts.get(2, 0) / counts.get(1, 0),
    }


snapshot_totals = {
    "illicit": int(wallet_counts["illicit"][1:].sum()),
    "licit": int(wallet_counts["licit"][1:].sum()),
    "unknown": int(wallet_counts["unknown"][1:].sum()),
}
snapshot_totals["labeled"] = snapshot_totals["illicit"] + snapshot_totals["licit"]

class_table = pd.DataFrame([
    class_distribution(txs_classes, "transactions"),
    class_distribution(wallets_classes, "wallets"),
])

snapshot_table = pd.DataFrame([{
    "entity": "wallet snapshots (deduplicated, per address-time step)",
    "total": wallet_distinct_rows,
    "illicit": snapshot_totals["illicit"],
    "licit": snapshot_totals["licit"],
    "unknown": snapshot_totals["unknown"],
    "labeled": snapshot_totals["labeled"],
    "labeled_pct": 100 * snapshot_totals["labeled"] / wallet_distinct_rows,
    "illicit_pct_of_all": 100 * snapshot_totals["illicit"] / wallet_distinct_rows,
    "illicit_pct_of_labeled": 100 * snapshot_totals["illicit"] / snapshot_totals["labeled"],
}])
snapshot_table["licit_per_illicit"] = snapshot_table["licit"] / snapshot_table["illicit"]

distribution = pd.concat([class_table, snapshot_table], ignore_index=True)
save_table("class_distribution.csv", distribution)
show(distribution)

In [ ]:
panels = [
    ("transactions", label_counts["transactions"]),
    ("wallets", label_counts["wallets"]),
    ("wallet snapshots", {"1": snapshot_totals["illicit"], "2": snapshot_totals["licit"],
                          "3": snapshot_totals["unknown"]}),
]
colors = {1: "#d62828", 2: "#2a9d8f", 3: "#adb5bd"}

fig, axes = plt.subplots(1, 3, figsize=(13, 4.2))
for ax, (title, counts) in zip(axes, panels):
    values = [int(counts.get(key, counts.get(str(key), 0))) for key in (1, 2, 3)]
    ax.bar([LABEL_NAMES[key] for key in (1, 2, 3)], values,
           color=[colors[key] for key in (1, 2, 3)])
    ax.set_yscale("log")
    ax.set_title(f"{title}\n(n = {sum(values):,}, log scale)")
    ax.set_ylabel("count")
    for position, value in enumerate(values):
        ax.text(position, value, f"{value:,}", ha="center", va="bottom", fontsize=8)
save_fig(fig, "class_balance.png")

## 6. Temporal Analysis

Time is part of the data model, not an afterthought: every later split, feature and graph
decision depends on how activity and labels move across the 49 steps. Three things matter here:

* **volume** — how much is there to learn from per step;
* **label coverage** — how much of each step is labeled at all;
* **illicit share** — how the positive class moves, because that is what the metric protocol has
  to survive.

Note the scale of the `unknown` class: most activity per step is unlabelled, which is why
"labeled" and "unknown" are never mixed below.

In [ ]:
txs_step = step_table(txs_counts, "transactions")
wallet_step = step_table(wallet_counts, "wallet_snapshots")
save_table("txs_temporal_by_step.csv", txs_step)
save_table("wallets_temporal_by_step.csv", wallet_step)

temporal_summary = pd.DataFrame([
    {
        "unit": unit,
        "total_min": int(frame["total"].min()), "total_max": int(frame["total"].max()),
        "labeled_min": int(frame["labeled"].min()), "labeled_max": int(frame["labeled"].max()),
        "labeled_pct_min": round(frame["labeled_pct_of_total"].min(), 2),
        "labeled_pct_max": round(frame["labeled_pct_of_total"].max(), 2),
        "illicit_min": int(frame["illicit"].min()), "illicit_max": int(frame["illicit"].max()),
        "illicit_pct_labeled_min": round(frame["illicit_pct_of_labeled"].min(), 2),
        "illicit_pct_labeled_median": round(frame["illicit_pct_of_labeled"].median(), 2),
        "illicit_pct_labeled_max": round(frame["illicit_pct_of_labeled"].max(), 2),
    }
    for unit, frame in (("transactions", txs_step), ("wallet snapshots", wallet_step))
])
save_table("temporal_summary.csv", temporal_summary)
show(temporal_summary)
show(txs_step.head(6))

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 7.5), sharex="col")
for column, (unit, frame) in enumerate((("transactions", txs_step), ("wallet snapshots", wallet_step))):
    top = axes[0, column]
    top.bar(frame["time_step"], frame["total"], color="#8ecae6", label="total")
    top.bar(frame["time_step"], frame["labeled"], color="#219ebc", label="labeled (1/2)")
    top.set_title(f"{unit}: activity per time step")
    top.set_ylabel("count")

    bottom = axes[1, column]
    bottom.plot(frame["time_step"], frame["illicit_pct_of_labeled"], marker="o", ms=3,
                lw=1, color="#d62828", label="illicit % of labeled")
    overall = 100 * frame["illicit"].sum() / frame["labeled"].sum()
    bottom.axhline(overall, ls="--", lw=1, color="grey", label=f"overall {overall:.2f}%")
    bottom.axvspan(EARLY_MAX + 0.5, N_STEPS + 0.5, color="#ffb703", alpha=0.15,
                   label="candidate future period (35-49)")
    bottom.set_xlabel("time step (1-49)")
    bottom.set_ylabel("% of labeled")
    bottom.legend(fontsize=8)
save_fig(fig, "temporal_activity.png")

## 7. Feature Analysis

The goal here is description, not selection. Four questions, in order of usefulness for the
modelling phases:

1. which features carry signal about the label (point-biserial `r` with illicit)?
2. how are the features distributed (range, skew, mass at zero) — i.e. what will need scaling?
3. which features are redundant copies of each other (in particular the 17 domain columns that
   Phase 1 flagged as near-duplicates of `Local_feature_*`)?
4. which features drift between the early and late periods — the covariate shift a temporal
   evaluation has to survive?

Quantiles, skewness and correlations come from the bounded systematic sample; means, standard
deviations and missingness are exact whole-file values.

In [ ]:
def feature_profile(stats: FeatureStats, summary: pd.DataFrame) -> pd.DataFrame:
    """Merge exact streaming statistics with sample-based quantiles and skewness."""
    sample = stats.sample()
    levels = [0.01, 0.25, 0.5, 0.75, 0.99]
    quantiles = sample.quantile(levels).T
    quantiles.columns = [f"p{int(level * 100)}" for level in levels]
    frame = summary.merge(quantiles, left_on="column", right_index=True, how="left")
    frame["skew"] = frame["column"].map(sample.skew())
    return frame


txs_profile = feature_profile(txs_stats, txs_quality)
wallet_profile = feature_profile(wallet_stats, wallet_quality)
save_table("txs_feature_summary.csv", txs_profile)
save_table("wallets_feature_summary.csv", wallet_profile)
print(f"transactions: {len(txs_profile)} features | wallets: {len(wallet_profile)} features")
show(txs_profile.head(6))

# The bounded sample drives all quantiles and correlations, so it should resemble the
# full file. A large mismatch would mean the systematic stride is biased.
full_share = class_table.set_index("entity")["illicit_pct_of_all"].to_dict()
for entity, stats in (("transactions", txs_stats), ("wallets", wallet_stats)):
    labels_sample = stats.sample_labels()
    sample_share = 100 * float(np.mean(labels_sample == 1))
    print(f"{entity}: sample of {labels_sample.size:,} rows has {sample_share:.2f}% illicit "
          f"vs {full_share[entity]:.2f}% illicit in the full file")

for entity, profile in (("transactions", txs_profile), ("wallets", wallet_profile)):
    ranked = profile.dropna(subset=["corr_with_illicit"]).copy()
    ranked = ranked.reindex(ranked["corr_with_illicit"].abs().sort_values(ascending=False).index)
    print(f"--- {entity}: strongest association with illicit (top 12) ---")
    show(ranked[["column", "count", "mean", "std", "min", "max", "zeros_pct", "skew", "corr_with_illicit"]].head(12))

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8.5))

for ax, (entity, profile) in zip(axes[0], (("transactions", txs_profile), ("wallets", wallet_profile))):
    ranked = profile.dropna(subset=["corr_with_illicit"]).copy()
    ranked = ranked.reindex(ranked["corr_with_illicit"].abs().sort_values(ascending=False).index).head(15)
    ranked = ranked.sort_values("corr_with_illicit")
    ax.barh(ranked["column"], ranked["corr_with_illicit"],
            color=np.where(ranked["corr_with_illicit"] > 0, "#d62828", "#2a9d8f"))
    ax.set_title(f"{entity}: r with illicit (top 15 by |r|)")
    ax.set_xlabel("point-biserial r  (positive = higher for illicit)")
    ax.tick_params(axis="y", labelsize=7)

for ax, (entity, profile) in zip(axes[1], (("transactions", txs_profile), ("wallets", wallet_profile))):
    drift = profile.dropna(subset=["drift_std"]).copy()
    drift = drift.reindex(drift["drift_std"].abs().sort_values(ascending=False).index).head(12)
    drift = drift.sort_values("drift_std")
    ax.barh(drift["column"], drift["drift_std"],
            color=np.where(drift["drift_std"] > 0, "#ffb703", "#457b9d"))
    ax.set_title(f"{entity}: temporal drift, steps 1-34 vs 35-49")
    ax.set_xlabel("change in mean, in standard deviations")
    ax.tick_params(axis="y", labelsize=7)

save_fig(fig, "feature_signal_and_drift.png")

In [ ]:
def redundancy(sample: pd.DataFrame, threshold: float = 0.99) -> pd.Series:
    """Pairs of features whose |correlation| reaches the threshold in the sample."""
    correlation = sample.corr().abs()
    upper = correlation.where(np.triu(np.ones(correlation.shape), k=1).astype(bool))
    pairs = upper.stack().sort_values(ascending=False)
    return pairs[pairs >= threshold]


txs_redundant = redundancy(txs_stats.sample())
wallet_redundant = redundancy(wallet_stats.sample())
print(f"transactions: {len(txs_redundant):,} feature pairs with |r| >= 0.99")
print(txs_redundant.head(8).to_string())
print(f"\nwallets: {len(wallet_redundant):,} feature pairs with |r| >= 0.99")
print(wallet_redundant.head(8).to_string())

# Are the 17 domain columns already encoded in the anonymous Local_feature_* block?
local_columns = [c for c in txs_features_all if c.startswith("Local_feature")]
domain_vs_local = (txs_stats.sample()[txs_domain_columns + local_columns].corr()
                   .loc[txs_domain_columns, local_columns]
                   .dropna(how="all"))
domain_redundancy = pd.DataFrame({
    "best_local_feature": domain_vs_local.abs().idxmax(axis=1),
    "abs_corr": domain_vs_local.abs().max(axis=1),
}).sort_values("abs_corr", ascending=False)
domain_redundancy.index.name = "domain_column"
save_table("txs_domain_redundancy.csv", domain_redundancy.reset_index())
print("\n17 domain columns vs their closest Local_feature_* counterpart:")
show(domain_redundancy)

## 8. Graph Analysis

Four directed edge types make up the heterogeneous graph, and they behave very differently:

* `tx->tx` — Bitcoin flow between transactions;
* `addr->tx` — addresses providing inputs to a transaction;
* `tx->addr` — addresses receiving outputs from a transaction;
* `addr->addr` — direct address-to-address flow, including multi-edges and self-transfers.

Edge lists are streamed and reduced to integer node codes before any matrix is built. Degrees,
self-loops, repeated pairs, reciprocal edges and weakly-connected components are computed per
edge type; the address↔transaction pair is also analysed as one combined bipartite graph, which
is what a heterogeneous model will actually see.

This is structural EDA only — no final training graph is constructed here.

In [ ]:
def ccdf(values, points: int = 300):
    """P(degree >= x), sampled on a log grid, for plotting heavy-tailed distributions."""
    values = np.asarray(values)
    positive = np.sort(values[values > 0])
    if positive.size == 0:
        return np.array([]), np.array([])
    grid = np.unique(np.geomspace(1, positive.max(), points).astype(np.int64))
    return grid, 1.0 - np.searchsorted(positive, grid, side="left") / positive.size


def edge_endpoints(path, source_column, target_column, index: pd.Index):
    """Stream an edge list into integer-coded endpoints, plus outside-universe counts."""
    source_parts, target_parts = [], []
    edges = outside_source = outside_target = 0
    for chunk in iter_chunks(path, usecols=[source_column, target_column]):
        source = index.get_indexer(chunk[source_column].astype(str).to_numpy())
        target = index.get_indexer(chunk[target_column].astype(str).to_numpy())
        outside_source += int((source < 0).sum())
        outside_target += int((target < 0).sum())
        source_parts.append(np.where(source < 0, 0, source).astype(np.int64))
        target_parts.append(np.where(target < 0, 0, target).astype(np.int64))
        edges += len(chunk)
        del chunk, source, target
    return (np.concatenate(source_parts), np.concatenate(target_parts), edges,
            outside_source, outside_target)


def graph_summary(edge_type: str, path, source_column: str, target_column: str, index: pd.Index) -> dict:
    """Degrees, multiplicities and weak components for one edge list."""
    node_count = len(index)
    source, target, edges, outside_source, outside_target = edge_endpoints(
        path, source_column, target_column, index)
    out_degree = np.bincount(source, minlength=node_count)
    in_degree = np.bincount(target, minlength=node_count)
    total_degree = out_degree + in_degree

    keys = source * node_count + target
    unique_keys = np.unique(keys)
    adjacency = csr_matrix((np.ones(source.size, dtype=np.int8), (source, target)),
                           shape=(node_count, node_count))
    n_components, components = connected_components(adjacency, directed=False)
    sizes = np.bincount(components)
    top = np.argsort(total_degree)[::-1][:10]
    grid, survival = ccdf(total_degree)

    return {
        "edge_type": edge_type,
        "nodes": node_count,
        "edges": edges,
        "unique_pairs": int(unique_keys.size),
        "repeated_pairs": int(edges - unique_keys.size),
        "self_loops": int((source == target).sum()),
        "reciprocal_edges": int(np.isin(target * node_count + source, unique_keys).sum()),
        "unique_sources": int(np.unique(source).size),
        "unique_targets": int(np.unique(target).size),
        "nodes_with_no_edge": int((total_degree == 0).sum()),
        "endpoints_outside_universe": outside_source + outside_target,
        "mean_degree": float(total_degree.mean()),
        "median_degree": float(np.median(total_degree)),
        "p99_degree": float(np.percentile(total_degree, 99)),
        "max_degree": int(total_degree.max()),
        "components": int(n_components),
        "largest_component": int(sizes.max()),
        "largest_component_pct": 100 * float(sizes.max()) / node_count,
        "top_nodes": [{"node": str(index[i]), "out_degree": int(out_degree[i]),
                       "in_degree": int(in_degree[i])} for i in top],
        "ccdf": {"x": grid, "y": survival},
    }

In [ ]:
universes = {
    "txs": pd.Index(txs_meta[TX_ID].to_numpy()),
    "wallets": pd.Index(wallets_classes[ADDRESS].to_numpy()),
    "combined": pd.Index(np.unique(np.concatenate([txs_meta[TX_ID].to_numpy(),
                                                   wallets_classes[ADDRESS].to_numpy()]))),
}

edge_results = {}
for edge_type, spec in EDGE_SPECS.items():
    edge_results[edge_type] = graph_summary(edge_type, DATA_DIR / FILES[spec["key"]],
                                            spec["source"], spec["target"], universes[spec["universe"]])
    block = edge_results[edge_type]
    print(f"{edge_type:10s} {block['edges']:>10,} edges | {block['nodes']:>9,} nodes in universe | "
          f"{block['components']:>7,} weak components | {block['endpoints_outside_universe']:>4,} endpoints outside")

graph_table = pd.DataFrame(edge_results.values())
save_table("graph_summary.csv", graph_table.drop(columns=["top_nodes", "ccdf"]))
show(graph_table.set_index("edge_type")[[
    "nodes", "edges", "unique_pairs", "repeated_pairs", "self_loops", "reciprocal_edges",
    "unique_sources", "unique_targets", "nodes_with_no_edge", "endpoints_outside_universe",
    "mean_degree", "median_degree", "p99_degree", "max_degree", "components", "largest_component",
]])

hubs = pd.concat(
    [pd.DataFrame(block["top_nodes"]).assign(edge_type=edge_type).head(5)
     for edge_type, block in edge_results.items()],
    ignore_index=True,
)
save_table("graph_hubs.csv", hubs)
show(hubs)

In [ ]:
# The address<->transaction edges seen together, as one bipartite graph.
source_parts, target_parts = [], []
for key, source_column, target_column in (("addr_tx", "input_address", TX_ID),
                                          ("tx_addr", TX_ID, "output_address")):
    source, target, *_ = edge_endpoints(DATA_DIR / FILES[key], source_column, target_column,
                                        universes["combined"])
    source_parts.append(source)
    target_parts.append(target)

combined_edges = int(sum(part.size for part in source_parts))
combined_graph = csr_matrix(
    (np.ones(combined_edges, dtype=np.int8),
     (np.concatenate(source_parts), np.concatenate(target_parts))),
    shape=(len(universes["combined"]),) * 2,
)
n_components, component_labels = connected_components(combined_graph, directed=False)
component_sizes = np.bincount(component_labels)
singletons = int((component_sizes == 1).sum())
check("combined address+transaction graph: singleton components", singletons, REFERENCE["txs_blank_rows"])
print(f"combined address<->transaction graph: {len(universes['combined']):,} nodes, "
      f"{n_components:,} weakly-connected components, largest {component_sizes.max():,} "
      f"({100 * component_sizes.max() / len(universes['combined']):.1f}%), {singletons:,} singletons")

del combined_graph, component_labels, component_sizes, source_parts, target_parts
gc.collect()

# Do transaction-graph components line up with time steps? If they do, transaction-only
# message passing can never cross a time-step boundary, which changes what a GNN can learn.
source, target, *_ = edge_endpoints(DATA_DIR / FILES["txs_edges"], "txId1", "txId2", universes["txs"])
tx_graph = csr_matrix((np.ones(source.size, dtype=np.int8), (source, target)),
                      shape=(len(universes["txs"]),) * 2)
tx_components = connected_components(tx_graph, directed=False)[1]
spans = (pd.DataFrame({"component": tx_components, "time_step": txs_meta[STEP].to_numpy()})
         .groupby("component")["time_step"].agg(["nunique", "min", "max"]))
within = int((spans["nunique"] == 1).sum())
print(f"\ntransaction graph: {len(spans):,} components, {within:,} confined to a single time step "
      f"({100 * within / len(spans):.1f}%)")
print(spans["nunique"].value_counts().sort_index().rename("components").to_string())
save_table("txs_component_time_span.csv", spans.reset_index())

del tx_graph, tx_components, source, target
gc.collect()

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 5))
for edge_type, block in edge_results.items():
    ax.plot(block["ccdf"]["x"], block["ccdf"]["y"], marker=".", ms=3, lw=1,
            label=f"{edge_type}  (universe {block['nodes']:,})")
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("total degree (in + out)")
ax.set_ylabel("P(degree >= x)")
ax.set_title("Degree survival by edge type")
ax.legend(fontsize=8)
save_fig(fig, "degree_ccdf.png")

## 9. Temporal Split Investigation

The proposed setup is **train on steps 1–34, test on steps 35–49**. It is investigated here, not
adopted: a split is only practical if the future period contains enough labelled positives to
measure anything, and if the prevalence gap between periods is small enough that the metric stays
interpretable.

Three questions drive the table below:

* how many labelled and illicit samples exist in each period (transactions and wallet snapshots)?
* can a validation window be carved out of the *historical* period, so the future stays untouched?
* how badly does illicit prevalence shift between the periods that would be compared?

No future information is used anywhere in this notebook: everything is descriptive statistics of
raw labels and features.

In [ ]:
WINDOW_TRAIN = "train 1-34 (proposed)"
WINDOW_TEST = "test 35-49 (proposed)"
WINDOW_VAL = "val 35-42"
WINDOW_TEST2 = "test 43-49"
WINDOWS = {
    WINDOW_TRAIN: (1, 34),
    WINDOW_TEST: (35, 49),
    WINDOW_VAL: (35, 42),
    WINDOW_TEST2: (43, 49),
    "validation inside history 25-34": (25, 34),
    "val 35-40": (35, 40),
    "test 41-49": (41, 49),
}
WINDOW_REPORT = [WINDOW_TRAIN, WINDOW_TEST, WINDOW_VAL, WINDOW_TEST2, "validation inside history 25-34",
                 "val 35-40", "test 41-49"]


def window_stats(frame: pd.DataFrame, start: int, end: int) -> dict:
    block = frame[(frame["time_step"] >= start) & (frame["time_step"] <= end)]
    labeled = int(block["labeled"].sum())
    illicit = int(block["illicit"].sum())
    return {
        "steps": f"{start}-{end}",
        "total": int(block["total"].sum()),
        "labeled": labeled,
        "illicit": illicit,
        "licit": int(block["licit"].sum()),
        "unknown": int(block["unknown"].sum()),
        "illicit_pct_of_labeled": (100 * illicit / labeled) if labeled else np.nan,
    }


windows = pd.DataFrame([
    {"unit": unit, "window": name, **window_stats(frame, *WINDOWS[name])}
    for unit, frame in (("transactions", txs_step), ("wallet snapshots", wallet_step))
    for name in WINDOW_REPORT
]).replace([np.inf, -np.inf], np.nan)
save_table("split_windows.csv", windows)
show(windows)

train_tx = windows[(windows["unit"] == "transactions") & (windows["window"] == WINDOW_TRAIN)].iloc[0]
test_tx = windows[(windows["unit"] == "transactions") & (windows["window"] == WINDOW_TEST)].iloc[0]
train_wallet = windows[(windows["unit"] == "wallet snapshots") & (windows["window"] == WINDOW_TRAIN)].iloc[0]
test_wallet = windows[(windows["unit"] == "wallet snapshots") & (windows["window"] == WINDOW_TEST)].iloc[0]
print(f"transactions  : train illicit {train_tx['illicit_pct_of_labeled']:.2f}% ({train_tx['illicit']:,} of "
      f"{train_tx['labeled']:,}) -> test {test_tx['illicit_pct_of_labeled']:.2f}% ({test_tx['illicit']:,} of "
      f"{test_tx['labeled']:,})")
print(f"wallet snaps  : train illicit {train_wallet['illicit_pct_of_labeled']:.2f}% "
      f"({train_wallet['illicit']:,}) -> test {test_wallet['illicit_pct_of_labeled']:.2f}% "
      f"({test_wallet['illicit']:,})")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
for ax, unit in zip(axes, ("transactions", "wallet snapshots")):
    block = windows[windows["unit"] == unit].iloc[::-1]
    ax.barh(block["window"], block["illicit_pct_of_labeled"], color="#d62828", alpha=0.85)
    for position, row in enumerate(block.itertuples()):
        ax.text(row.illicit_pct_of_labeled, position,
                f"  {row.illicit:,} illicit / {row.labeled:,} labeled", va="center", fontsize=7)
    ax.set_title(f"{unit}: illicit share of labeled, by candidate window")
    ax.set_xlabel("illicit % of labeled")
    ax.tick_params(axis="y", labelsize=8)
save_fig(fig, "split_windows.png")

## 10. Key Findings

Everything this notebook currently knows is printed by the cell below — a machine-generated
digest of the numbers that were just computed, plus the result of every cross-check against the
Phase 1 inventory.

**The prose summary is written after the run, not before.** Read the digest and the figures, then
fill in the points below with the observed values (the digest file `eda_digest.txt` and the
tables in `OUT_DIR` are there to copy from):

* dataset scale and integrity — did every check pass, and did anything disagree with Phase 1?
* data-quality corrections worth carrying forward (wallet duplicate rows, blank domain columns,
  redundant features, missing addresses);
* class imbalance — how severe, and how much it moves between time steps;
* feature signal and drift — which features separate illicit from licit, and which shift over time;
* graph structure — hub behaviour, multi-edges, self-loops, and how fragmented each edge type is;
* whether the transaction graph is time-step-aligned, and what that implies for message passing;
* temporal split — how much labelled/illicit data each candidate window actually contains.

If any check printed `!!`, treat it as a finding to explain before building any model on top of
this data.

In [ ]:
tx_row = overview.iloc[0]
wallet_row = overview.iloc[1]
digest = f"""
BitcoinGraphGuard - Phase 1 EDA digest ({time.strftime('%Y-%m-%d %H:%M')})
----------------------------------------------------------------------------------
transactions         : {len(txs_classes):,} nodes | {len(txs_features_all)} features ({len(txs_domain_columns)} domain)
wallet addresses     : {len(wallets_classes):,} nodes | {len(wallets_features_all)} features
wallet rows          : {wallet_raw_rows:,} raw -> {wallet_distinct_rows:,} distinct snapshots ({wallet_duplicate_rows:,} duplicates dropped)
transaction labels   : illicit {tx_row['illicit']:,} = {tx_row['illicit_pct_of_all']:.2f}% of all / {tx_row['illicit_pct_of_labeled']:.2f}% of labeled
wallet labels        : illicit {wallet_row['illicit']:,} = {wallet_row['illicit_pct_of_all']:.2f}% of all / {wallet_row['illicit_pct_of_labeled']:.2f}% of labeled
wallet snapshots     : illicit {snapshot_totals['illicit']:,} = {100 * snapshot_totals['illicit'] / wallet_distinct_rows:.2f}% of snapshots
illicit % by step    : transactions {txs_step['illicit_pct_of_labeled'].min():.2f}-{txs_step['illicit_pct_of_labeled'].max():.2f}% | wallet snapshots {wallet_step['illicit_pct_of_labeled'].min():.2f}-{wallet_step['illicit_pct_of_labeled'].max():.2f}%
edges                : {', '.join(f'{row.edge_type} {row.rows:,}' for row in edge_inventory.itertuples())}
address graph        : {edge_results['addr->addr']['components']:,} components, largest {edge_results['addr->addr']['largest_component_pct']:.1f}%
transaction graph    : {len(spans):,} components, {within:,} confined to one time step
combined addr<->tx   : {singletons:,} singleton components
redundant features   : |r| >= 0.99 pairs -> transactions {len(txs_redundant):,}, wallets {len(wallet_redundant):,}
blank domain rows    : {len(blank_tx_ids):,} transactions
temporal windows     : train 1-34 illicit {train_tx['illicit_pct_of_labeled']:.2f}% ({train_tx['illicit']:,}) vs test 35-49 {test_tx['illicit_pct_of_labeled']:.2f}% ({test_tx['illicit']:,})
wallet 1-34 vs 35-49 : illicit {train_wallet['illicit_pct_of_labeled']:.2f}% vs {test_wallet['illicit_pct_of_labeled']:.2f}%
"""

(OUT_DIR / "eda_digest.txt").write_text(digest, encoding="utf-8")
save_table("checks.csv", pd.DataFrame(CHECKS))
save_json("eda_summary.json", {
    "digest": digest,
    "checks": CHECKS,
    "windows": windows.to_dict(orient="records"),
    "graph": {name: {k: v for k, v in block.items() if k != "ccdf"} for name, block in edge_results.items()},
})

failed = [row for row in CHECKS if not row["ok"]]
print(digest)
print(f"checks: {len(CHECKS) - len(failed)}/{len(CHECKS)} passed")
for row in failed:
    print(f"  !! {row['check']}: got {row['value']}, expected {row['expected']}")
print(f"\nartifacts written to {OUT_DIR}")

txs_stats.release()
wallet_stats.release()

## 11. Decisions for Next Phase

Nothing below is decided yet — these are the decisions the EDA exists to inform, and each one
should now be resolved from the results above. Do not start the XGBoost notebook until they are.

**XGBoost feature set**
* keep all 182 transaction features, or drop the 17 domain columns that duplicate `Local_feature_*`
  (check `txs_domain_redundancy.csv`)?
* how to represent the 965 address-less transactions — drop them, or keep them with a
  `has_address_links` indicator and masked domain columns (never treat blanks as `0`)?
* which wallet aggregates to attach to transactions without leaking future time steps?

**Class imbalance and metrics**
* weight strategy (`scale_pos_weight` vs resampling) given the observed illicit share;
* metric protocol: PR-AUC as primary, plus precision/recall/F1 at a fixed operating point, and
  always reported per temporal window because prevalence is non-stationary.

**Temporal split and validation**
* which candidate window in `split_windows.csv` becomes train/validation/test, and whether
  validation is carved from history rather than from the future;
* whether the low-prevalence future window is accepted as the realistic deployment condition, and
  what that means for reporting;
* wallet labels are per address and reused across snapshots — decide the unit of evaluation
  (address vs snapshot) before the first result is published.

**Graph construction and GNN training**
* deduplicate wallet snapshots to `(address, time step)` before any split;
* keep the graph heterogeneous (four edge types) rather than flattening it;
* use mini-batch neighbour sampling — the address component is effectively one giant component,
  so full-graph training is not an option;
* if the transaction graph is time-step-aligned, transaction-only message passing cannot cross
  time steps: the report generated in §8 decides whether RGCN/HGT must rely on address edges for
  cross-step signal.

**Next phase**
* once the above are reviewed, the next phase is `notebooks/02_xgboost.ipynb`: the classical
  baseline on engineered tabular features, under the same temporal protocol — no GNN work starts
  before that baseline exists.